# 너비 우선 탐색 (Breadth-First Search)

`-` 시작 노드에서 가까운 노드부터 차례대로 탐색하는 알고리즘

## 보물섬

- 문제 출처: [정올 1462번](https://jungol.co.kr/problem/1462)

`-` 각 육지 칸마다 BFS를 수행해서 가장 먼 육지 칸까지의 이동 시간을 계산하자

`-` 이들 중 최댓값이 문제의 정답이 된다

`-` 전체 알고리즘의 시간 복잡도는 모든 칸이 육지인 최악의 경우 $O\left(R^2C^2\right)$이다

In [6]:
from collections import deque
from itertools import product


def find_land_positions(graph, n_rows, n_cols):
    return [(r, c) for r, c in product(range(n_rows), range(n_cols)) if graph[r][c] == LAND]


def bfs(graph, r_start, c_start, n_rows, n_cols):
    pos_start = r_start, c_start
    queue = deque([pos_start])
    visited = {pos_start: 0}
    drc = [(0, -1), (0, 1), (-1, 0), (1, 0)]
    while queue:
        r, c = queue.popleft()
        for dr, dc in drc:
            nr, nc = r + dr, c + dc
            is_in_range = 0 <= nr < n_rows and 0 <= nc < n_cols
            if not is_in_range or graph[nr][nc] != LAND:
                continue
            if (nr, nc) in visited:
                continue
            visited[nr, nc] = visited[r, c] + 1
            queue.append((nr, nc))
    return visited


def solution():
    global LAND
    R, C = map(int, input().split())
    graph = [input() for _ in range(R)]
    LAND = "L"
    land_positions = find_land_positions(graph, R, C)
    answer = 0
    for r, c in land_positions:
        pos2time = bfs(graph, r, c, R, C)
        time = max(pos2time.values())
        answer = max(time, answer)
    print(answer)


solution()

# input
# 2 2
# LL
# LL

 2 2
 LL
 LL


2


## 치즈

- 문제 출처: [정올 1840번](https://jungol.co.kr/problem/1840)

`-` 0-1 BFS로 풀 수 있다고 한다

`-` 치즈가 외부 공기와 접촉하면 한 시간 후에 녹아 없어진다

`-` 공기로 이동할 땐 가중치를 $0$으로 두고 치즈로 이동할 땐 가중치를 $1$로 두자

`-` 그럼 공기로만 이루어진 컴포넌트를 $0$ 시간에 구성한 뒤 치즈와의 접촉 여부를 판단할 수 있게 된다

`-` 시작 지점은 항상 외부 공기임이 보증된 $(0,0)$이며 가중치가 $0$과 $1$뿐이니 0-1 BFS를 사용하여 $O(V+E)$에 해결할 수 있다

`-` 거리 배열의 최댓값이 치즈가 모두 녹아 없어지는 데 걸리는 시간이며 치즈가 있으면서 거리가 최댓값인 칸의 개수가 녹기 직전 칸의 개수이다

In [7]:
from collections import deque


def bfs(graph, source):
    n_rows, n_cols = len(graph), len(graph[0])
    queue = deque([source])
    visited = [[-1] * n_cols for _ in range(n_rows)]
    visited[source[0]][source[1]] = 0
    drc = [(-1, 0), (1, 0), (0, -1), (0, 1)]
    while queue:
        r, c = queue.popleft()
        for dr, dc in drc:
            nr, nc = r + dr, c + dc
            is_in_range = 0 <= nr < n_rows and 0 <= nc < n_cols
            if not is_in_range or visited[nr][nc] >= 0:
                continue
            is_cheese = graph[nr][nc] == 1
            if is_cheese:
                queue.append((nr, nc))
                visited[nr][nc] = visited[r][c] + 1
            else:
                queue.appendleft((nr, nc))
                visited[nr][nc] = visited[r][c]
    return visited


def solution():
    R, C = map(int, input().split())
    graph = [list(map(int, input().split())) for _ in range(R)]
    source = 0, 0
    visited = bfs(graph, source)
    end_time = max(map(max, visited))
    count = sum(1 for r in range(R) for c in range(C) if visited[r][c] == end_time and graph[r][c] == 1)
    print(end_time)
    print(count)


solution()

# input
# 3 3
# 0 0 0
# 0 1 0
# 0 0 0

 3 3
 0 0 0
 0 1 0
 0 0 0


1
1


`-` 근데 꼭 가중치가 $0$과 $1$일 필요는 없고 $0$과 $c$여도 괜찮다

`-` 하지만 $0$이 아닌 $a,b$라면 가중치가 $a$인 간선을 우선 사용하는 게 항상 최단 거리를 보장하는 게 아니게 된다

## 치즈

- 문제 출처: [정올 1870번](https://jungol.co.kr/problem/1870)

`-` 이전 [치즈](https://jungol.co.kr/problem/1840) 문제와의 차이점은 치즈가 있는 칸의 $2$변 이상이 외부 공기와 접촉해야 치즈가 녹는다는 것이다 (이전 문제에선 $1$변)

`-` 0-1 BFS를 수행하며 치즈로 이동할 때 카운팅을 하자. 만약 카운팅이 $2$가 됐다면 큐에 넣고 방문 표시를 하면 된다

In [8]:
from collections import deque


def bfs(graph, source):
    n_rows, n_cols = len(graph), len(graph[0])
    queue = deque([source])
    visited = [[-1] * n_cols for _ in range(n_rows)]
    visited[source[0]][source[1]] = 0
    counts = [[0] * n_cols for _ in range(n_rows)]
    drc = [(-1, 0), (1, 0), (0, -1), (0, 1)]
    while queue:
        r, c = queue.popleft()
        for dr, dc in drc:
            nr, nc = r + dr, c + dc
            is_in_range = 0 <= nr < n_rows and 0 <= nc < n_cols
            if not is_in_range or visited[nr][nc] >= 0:
                continue
            is_cheese = graph[nr][nc] == 1
            if is_cheese:
                counts[nr][nc] += 1
                if counts[nr][nc] < 2:
                    continue
                queue.append((nr, nc))
                visited[nr][nc] = visited[r][c] + 1
            else:
                queue.appendleft((nr, nc))
                visited[nr][nc] = visited[r][c]
    return visited


def solution():
    R, C = map(int, input().split())
    graph = [list(map(int, input().split())) for _ in range(R)]
    source = 0, 0
    visited = bfs(graph, source)
    end_time = max(map(max, visited))
    print(end_time)


solution()

# input
# 3 3
# 0 0 0
# 0 1 0
# 0 0 0

 3 3
 0 0 0
 0 1 0
 0 0 0


1


## 화염에서탈출

- 문제 출처: [정올 1082번](https://jungol.co.kr/problem/1082)

`-` 불이든 재우든 `S`, `*`, `X`인 칸에는 갈 필요가 없다

`-` 불이 먼저 이동한 뒤 재우가 이동한다. 따라서 불의 좌표와 재우가 처음 서 있는 위치를 큐에 넣자. 불이 큐에서 먼저 나오면서 그래프를 갱신하게 된다

`-` 큐에서 나온 원소가 재운인 경우 거리 배열을 갱신하고 다음에 이동할 칸을 `S`로 갱신하자

`-` 불인 경우 다음에 이동할 칸만 `*`로 갱신하자

`-` 같은 좌표를 $2$번 이상 큐에 넣지 않으므로 전체 알고리즘의 시간 복잡도는 $O(RC)$이다

`-` 만약 재우가 움직일 때마다 불이 옮기는 것을 반영하고자 이중 for문을 돌며 인접한 $4$칸 중 최소 한 곳에 불이 있는지 확인하는 건 최악의 경우 $O\left(R^2 C^2\right)$의 시간 복잡도를 가진다

In [2]:
from collections import deque


def find_positions(graph, mark):
    n_rows, n_cols = len(graph), len(graph[0])
    return [(r, c) for r in range(n_rows) for c in range(n_cols) if graph[r][c] == mark]


def bfs(graph, source, sink, flames):
    n_rows, n_cols = len(graph), len(graph[0])
    queue = deque(flames + [source])
    distances = [[-1] * n_cols for _ in range(n_rows)]
    distances[source[0]][source[1]] = 0
    drc = [(-1, 0), (1, 0), (0, -1), (0, 1)]
    while queue:
        r, c = queue.popleft()
        for dr, dc in drc:
            nr, nc = r + dr, c + dc
            is_in_range = 0 <= nr < n_rows and 0 <= nc < n_cols
            if not is_in_range:
                continue
            if graph[nr][nc] != "D" and graph[nr][nc] != ".":
                continue
            is_man = graph[r][c] == "S"
            if is_man:
                queue.append((nr, nc))
                graph[nr][nc] = "S"
                distances[nr][nc] = distances[r][c] + 1
            elif graph[nr][nc] == ".":
                queue.append((nr, nc))
                graph[nr][nc] = "*"
    if distances[sink[0]][sink[1]] == -1:
        return "impossible"
    return distances[sink[0]][sink[1]]


def solution():
    R, C = map(int, input().split())
    graph = [list(input().rstrip()) for _ in range(R)]
    source = find_positions(graph, "S")[0]
    sink = find_positions(graph, "D")[0]
    flames = find_positions(graph, "*")
    answer = bfs(graph, source, sink, flames)
    print(answer)


solution()

# input
# 2 2
# DS
# ..

 2 2
 DS
 ..


1
